In [42]:
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda


In [43]:
BATCH_SIZE = 64
TOTAL_EPOCHS = 100
LR = 0.01
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 0   

DATA_DIR = "./data"


CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

print("Batch size:", BATCH_SIZE)
print("Epochs:", TOTAL_EPOCHS)
print("Learning rate:", LR)


Batch size: 64
Epochs: 100
Learning rate: 0.01


In [44]:
import torchvision.transforms as T

train_tfms = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_tfms = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])



In [45]:
from torch.utils.data import DataLoader
import torchvision

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=train_tfms
)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=test_tfms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS
)

classes = train_dataset.classes
print("Classes:", classes)
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train samples: 50000
Test samples: 10000


In [46]:
import torch.nn as nn
from torchvision.models import resnet18

model = resnet18(weights=None)   
model.fc = nn.Linear(model.fc.in_features, 10)

print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [47]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TOTAL_EPOCHS
)

print("Loss, optimizer, scheduler ready")


Loss, optimizer, scheduler ready


In [48]:
@torch.no_grad()
def evaluate():
    model.eval()   

    correct = 0
    total = 0

    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return accuracy


In [ ]:
def train_one_epoch():
    model.train()   #switch to training mode

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1) reset gradients
        optimizer.zero_grad()

        # 2) forward pass
        outputs = model(images)

        # 3) compute loss
        loss = criterion(outputs, labels)

        # 4) backward pass
        loss.backward()

        # 5) update weights
        optimizer.step()

        # statistics
        running_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    train_accuracy = correct / total

    return avg_loss, train_accuracy


In [50]:
best_test_acc = 0.0

for epoch in range(1, TOTAL_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()

    scheduler.step()   

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), "best_resnet_cifar10.pt")

    print(
        f"Epoch {epoch:03d}/{TOTAL_EPOCHS} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Test Acc: {test_acc*100:.2f}% | "
        f"Best: {best_test_acc*100:.2f}%"
    )

print("Training finished. Best model saved.")



RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [17]:
import torch

model.load_state_dict(torch.load("best_resnet_cifar10.pt", map_location=device))
model = model.to(device)

print("Loaded best checkpoint.")
print("Test acc now:", evaluate() * 100)


Loaded best checkpoint.
Test acc now: 86.35000000000001


In [18]:
import torch.optim as optim
import torch.nn as nn

# helps generalization a bit
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# KEY: low learning rate so we refine pretrained features instead of destroying them
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

FINE_TUNE_EPOCHS = 40
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINE_TUNE_EPOCHS)

best_test_acc = 0.0
print("Fine-tuning setup ready")


Fine-tuning setup ready


In [53]:
FINE_TUNE_EPOCHS = 40


In [22]:
model = model.to(device)
print("Model device:", next(model.parameters()).device)


Model device: cuda:0


In [23]:
import torch
device = torch.device("cuda")
print("Device set to:", device)


Device set to: cuda


In [24]:
model = model.to(device)

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

print("Model:", next(model.parameters()).device)
print("Images:", images.device)
print("Labels:", labels.device)


Model: cuda:0
Images: cuda:0
Labels: cuda:0


In [40]:
model = model.to(device)
print("model param device:", next(model.parameters()).device)


model param device: cuda:0


In [41]:
images, labels = next(iter(train_loader))
images = images.to(device)
out = model(images)
print("forward ok:", out.shape, out.device)


forward ok: torch.Size([64, 10]) cuda:0


In [56]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = model.to(device)
print("Model device:", next(model.parameters()).device)


Using device: cuda
Model device: cuda:0


In [61]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

FINE_TUNE_EPOCHS = 100
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINE_TUNE_EPOCHS)

print("Optimizer ready. First param device:", optimizer.param_groups[0]["params"][0].device)


Optimizer ready. First param device: cuda:0


In [58]:
images, labels = next(iter(train_loader))
images = images.to(device)
out = model(images)
print("Forward OK:", out.shape, out.device)


Forward OK: torch.Size([64, 10]) cuda:0


In [62]:
for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()
    scheduler.step()

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), "best_resnet_cifar10_ft.pt")

    print(
        f"FT {epoch:02d}/{FINE_TUNE_EPOCHS} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Test Acc: {test_acc*100:.2f}% | "
        f"Best: {best_test_acc*100:.2f}%"
    )

print("Done. Best fine-tuned model saved as best_resnet_cifar10_ft.pt")


FT 01/100 | Train Acc: 70.55% | Test Acc: 76.72% | Best: 84.80%
FT 02/100 | Train Acc: 70.82% | Test Acc: 78.12% | Best: 84.80%
FT 03/100 | Train Acc: 71.26% | Test Acc: 79.15% | Best: 84.80%
FT 04/100 | Train Acc: 72.05% | Test Acc: 80.49% | Best: 84.80%
FT 05/100 | Train Acc: 72.15% | Test Acc: 80.55% | Best: 84.80%
FT 06/100 | Train Acc: 72.25% | Test Acc: 79.18% | Best: 84.80%
FT 07/100 | Train Acc: 72.66% | Test Acc: 81.28% | Best: 84.80%
FT 08/100 | Train Acc: 73.15% | Test Acc: 79.63% | Best: 84.80%
FT 09/100 | Train Acc: 73.02% | Test Acc: 79.14% | Best: 84.80%
FT 10/100 | Train Acc: 73.56% | Test Acc: 80.29% | Best: 84.80%
FT 11/100 | Train Acc: 73.69% | Test Acc: 80.70% | Best: 84.80%
FT 12/100 | Train Acc: 73.79% | Test Acc: 80.47% | Best: 84.80%
FT 13/100 | Train Acc: 74.11% | Test Acc: 81.15% | Best: 84.80%
FT 14/100 | Train Acc: 74.09% | Test Acc: 81.01% | Best: 84.80%
FT 15/100 | Train Acc: 74.41% | Test Acc: 81.02% | Best: 84.80%
FT 16/100 | Train Acc: 74.77% | Test Acc

In [67]:
import torch

model.load_state_dict(torch.load("best_resnet_cifar10_ft.pt", map_location=device))
model = model.to(device)

print("Loaded FT checkpoint.")
print("Test acc now:", evaluate() * 100)


Loaded FT checkpoint.
Test acc now: 89.52


In [69]:
import torch.optim as optim

optimizer = optim.SGD(
    model.parameters(),
    lr=0.001,              
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

POLISH_EPOCHS = 200
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=POLISH_EPOCHS)

best_polish = 0.0
print("Polish phase ready")


Polish phase ready


In [70]:
for epoch in range(1, POLISH_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()
    scheduler.step()

    if test_acc > best_polish:
        best_polish = test_acc
        torch.save(model.state_dict(), "best_resnet_cifar10_polish.pt")

    print(
        f"POLISH {epoch:02d}/{POLISH_EPOCHS} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Test Acc: {test_acc*100:.2f}% | "
        f"Best: {best_polish*100:.2f}%"
    )

print("Polish finished. Saved best_resnet_cifar10_polish.pt")


POLISH 01/200 | Train Acc: 88.89% | Test Acc: 88.27% | Best: 88.27%
POLISH 02/200 | Train Acc: 88.21% | Test Acc: 88.31% | Best: 88.31%
POLISH 03/200 | Train Acc: 88.16% | Test Acc: 87.78% | Best: 88.31%
POLISH 04/200 | Train Acc: 87.90% | Test Acc: 88.12% | Best: 88.31%
POLISH 05/200 | Train Acc: 88.05% | Test Acc: 88.02% | Best: 88.31%
POLISH 06/200 | Train Acc: 88.09% | Test Acc: 87.85% | Best: 88.31%
POLISH 07/200 | Train Acc: 88.23% | Test Acc: 88.12% | Best: 88.31%
POLISH 08/200 | Train Acc: 88.14% | Test Acc: 87.97% | Best: 88.31%
POLISH 09/200 | Train Acc: 87.97% | Test Acc: 88.21% | Best: 88.31%
POLISH 10/200 | Train Acc: 88.04% | Test Acc: 88.30% | Best: 88.31%
POLISH 11/200 | Train Acc: 87.90% | Test Acc: 88.12% | Best: 88.31%
POLISH 12/200 | Train Acc: 87.87% | Test Acc: 88.07% | Best: 88.31%
POLISH 13/200 | Train Acc: 87.80% | Test Acc: 88.04% | Best: 88.31%
POLISH 14/200 | Train Acc: 87.88% | Test Acc: 88.26% | Best: 88.31%
POLISH 15/200 | Train Acc: 88.05% | Test Acc: 87

In [71]:
import torch

model.load_state_dict(torch.load("best_resnet_cifar10_polish.pt", map_location=device))
model = model.to(device)

acc = evaluate() * 100
print("Loaded best polish model. Test acc:", acc)


Loaded best polish model. Test acc: 90.07


In [72]:
@torch.no_grad()
def predict_tta(images):
    model.eval()
    logits1 = model(images)
    logits2 = model(torch.flip(images, dims=[3]))  # horizontal flip
    return (logits1 + logits2) / 2

@torch.no_grad()
def evaluate_tta():
    model.eval()
    correct, total = 0, 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = predict_tta(images)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

print("TTA test acc:", evaluate_tta() * 100)


TTA test acc: 91.2
